In [3]:
%%capture
import os
import pprint
import pandas as pd
from convokit import Corpus, download
from pathlib import Path
from datasets import load_dataset
import janitor

hf_token = os.getenv('HF_TOKEN')

In [ ]:
# load gap data from convokitA
gap = Corpus(filename = download("gap-corpus"))

convo_df = gap.get_conversations_dataframe().clean_names().reset_index(drop = False).drop(columns=['vectors'])
speaker_df = gap.get_speakers_dataframe().clean_names().reset_index(drop = False).drop(columns=['vectors'])
utt_df = gap.get_utterances_dataframe().clean_names().reset_index(drop = False).drop(columns=['vectors'])

Dataset already exists at /Users/jasminec/.convokit/saved-corpora/gap-corpus


In [35]:
ut_group_map  = (
    convo_df[['id', 'meta_group_number', 'meta_meeting_size', 'meta_meeting_length_in_minutes']]
    .rename(columns={
        'id': 'conversation_id',
        'meta_group_number': 'group_id',
        'meta_meeting_size': 'num_participants',
        'meta_meeting_length_in_minutes': 'duration'
        })
)

# merge with utterances 
utt_df.merge(ut_group_map, on='conversation_id', how='left')

,id,timestamp,text,speaker,reply_to,conversation_id,meta_end,meta_duration,meta_sentiment,meta_decision,meta_private,meta_survival_item,group_id,num_participants,duration
0,1.Pink.1,00:02.0,"""So what did everyone do as one""",1.Pink,None,1.Pink.1,00:03.5,00:01.5,NaN,NaN,NaN,NaN,1,3.0,9.75
1,1.Blue.1,00:04.0,"""I did uh cigarette lighter""",1.Blue,1.Pink.1,1.Pink.1,00:05.7,00:01.7,NaN,NaN,Private,Cigarette Lighter,1,3.0,9.75
2,1.Blue.2,00:06.4,"""For one""",1.Blue,1.Blue.1,1.Pink.1,00:07.2,00:00.7,NaN,NaN,Private,Cigarette Lighter,1,3.0,9.75
3,1.Pink.2,00:07.3,"""Mm okay I did knife""",1.Pink,1.Blue.2,1.Pink.1,00:09.3,00:02.1,NaN,NaN,Private,Knife,1,3.0,9.75
4,1.Green.1,00:09.4,"""Knife""",1.Green,1.Pink.2,1.Pink.1,00:09.9,00:00.6,NaN,NaN,NaN,Knife,1,3.0,9.75
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8004,9.Blue.47,06:26.1,"""$""",9.Blue,9.Blue.46,9.Green.1,06:26.5,00:00.4,Positive,NaN,NaN,NaN,9,3.0,6.63
8005,9.Green.114,06:29.2,"""But I dont know""",9.Green,9.Blue.47,9.Green.1,06:30.6,00:01.4,NaN,NaN,NaN,NaN,9,3.0,6.63
8006,9.Pink.102,06:30.2,"""$""",9.Pink,9.Green.114,9.Green.1,06:31.0,00:00.8,Positive,NaN,NaN,NaN,9,3.0,6.63
8007,9.Green.115,06:30.6,"""I just have a disposition against sectional m...",9.Green,9.Pink.102,9.Green.1,06:33.1,00:02.5,Negative,NaN,NaN,Air Map,9,3.0,6.63


In [14]:
utt_df

,id,timestamp,text,speaker,reply_to,conversation_id,meta_end,meta_duration,meta_sentiment,meta_decision,meta_private,meta_survival_item,vectors
0,1.Pink.1,00:02.0,"""So what did everyone do as one""",1.Pink,None,1.Pink.1,00:03.5,00:01.5,NaN,NaN,NaN,NaN,[]
1,1.Blue.1,00:04.0,"""I did uh cigarette lighter""",1.Blue,1.Pink.1,1.Pink.1,00:05.7,00:01.7,NaN,NaN,Private,Cigarette Lighter,[]
2,1.Blue.2,00:06.4,"""For one""",1.Blue,1.Blue.1,1.Pink.1,00:07.2,00:00.7,NaN,NaN,Private,Cigarette Lighter,[]
3,1.Pink.2,00:07.3,"""Mm okay I did knife""",1.Pink,1.Blue.2,1.Pink.1,00:09.3,00:02.1,NaN,NaN,Private,Knife,[]
4,1.Green.1,00:09.4,"""Knife""",1.Green,1.Pink.2,1.Pink.1,00:09.9,00:00.6,NaN,NaN,NaN,Knife,[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...
8004,9.Blue.47,06:26.1,"""$""",9.Blue,9.Blue.46,9.Green.1,06:26.5,00:00.4,Positive,NaN,NaN,NaN,[]
8005,9.Green.114,06:29.2,"""But I dont know""",9.Green,9.Blue.47,9.Green.1,06:30.6,00:01.4,NaN,NaN,NaN,NaN,[]
8006,9.Pink.102,06:30.2,"""$""",9.Pink,9.Green.114,9.Green.1,06:31.0,00:00.8,Positive,NaN,NaN,NaN,[]
8007,9.Green.115,06:30.6,"""I just have a disposition against sectional m...",9.Green,9.Pink.102,9.Green.1,06:33.1,00:02.5,Negative,NaN,NaN,Air Map,[]


In [36]:
from utils.utils import build_eval_examples_from_utterances, parse_first_int

# attach conversation-level speaker-count target (meeting size) to utterance rows
utt_eval_df = utt_df.merge(
    convo_df[["id", "meta_meeting_size"]].rename(
        columns={"id": "conversation_id", "meta_meeting_size": "ground_truth_speakers"}
    ),
    on="conversation_id",
    how="left",
)

# build eval examples in the same structure used in scripts/evals.ipynb
excerpts = build_eval_examples_from_utterances(utt_eval_df)

# fill ground truth for each excerpt
ground_truth_map = (
    utt_eval_df[["conversation_id", "ground_truth_speakers"]]
    .drop_duplicates()
    .set_index("conversation_id")["ground_truth_speakers"]
    .to_dict()
)

for e in excerpts:
    gt = ground_truth_map.get(e["source"])
    e["ground_truth_speakers"] = int(gt) if pd.notna(gt) else len(e["speakers"])

len(excerpts)

28

In [37]:
# quick sanity check
pd.DataFrame(
    [
        {
            "source": e["source"],
            "num_turns": e["num_turns"],
            "num_speakers": len(e["speakers"]),
            "ground_truth_speakers": e["ground_truth_speakers"],
        }
        for e in excerpts
    ]
).sort_values("source").reset_index(drop=True).head(10)

,source,num_turns,num_speakers,ground_truth_speakers
0,1.Pink.1,271,3,3
1,10.Orange.1,83,2,2
2,11.Pink.1,180,2,2
3,12.Blue.1,132,4,4
4,13.Yellow.1,169,2,2
5,14.Red.1,98,4,4
6,15.Orange.1,492,4,4
7,16.Yellow.1,118,3,3
8,17.Orange.1,253,4,4
9,18.Pink.1,279,3,3


In [38]:
# optional: speaker-count eval routine (mirrors scripts/evals.ipynb)
run_model_eval = False

if run_model_eval:
    import gc
    from textwrap import dedent
    import torch
    from nnsight import LanguageModel

    models = [
        "allenai/Olmo-3-1125-32B",
    ]

    num_eval_examples = min(10, len(excerpts))
    results = []

    for model_id in models:
        llm = LanguageModel(model_id, device_map="auto", remote=True)

        for e in excerpts[:num_eval_examples]:
            prompt = dedent(f"""\
            You will be asked how many speakers are involved in a conversation excerpt.
            Output only a single number as your final answer.

            Here is the excerpt:
            ```
            {e['unlabeled_text']}
            ```
            Output:
            """)

            response = ""
            error = None

            try:
                with llm.generate(
                    prompt,
                    max_new_tokens=64,
                    temperature=0.0,
                    do_sample=False,
                    remote=True,
                ):
                    generated = llm.generator.output.save()

                decoded = llm.tokenizer.decode(generated[0], skip_special_tokens=True)
                response = decoded[len(prompt):].strip() if decoded.startswith(prompt) else decoded.strip()
            except Exception as exc:
                error = str(exc)

            pred = parse_first_int(response)
            ground_truth = e["ground_truth_speakers"]

            results.append(
                {
                    "model": model_id,
                    "source": e["source"],
                    "predicted_speakers": pred,
                    "ground_truth": ground_truth,
                    "correct": pred == ground_truth,
                    "response": response,
                    "error": error,
                }
            )

        del llm
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    results_df = pd.DataFrame(results)
    display(results_df)
    if not results_df.empty:
        display(results_df.groupby("model", as_index=False)["correct"].mean().rename(columns={"correct": "accuracy"}))
else:
    print("Set run_model_eval=True to run the speaker-count eval routine.")

Set run_model_eval=True to run the speaker-count eval routine.
